# 1. Initialization

In [ ]:
import sys
import os

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(repo_root)

print(f"[INFO] Repo root added to path: {repo_root}")

In [ ]:
print("[INFO] Starting DAILY Bronze pipeline")

from pyspark.sql import SparkSession

from common.postgres_ingest import PostgresBronzeIngest
from common.landing_to_bronze import LandingToBronzeIngest

spark = SparkSession.builder.getOrCreate()
batch_id = dbutils.widgets.get("batch_id")

# 2. Landing → Bronze

In [ ]:
landing_entities = [
    "users",
    "subscriptions",
    "subscription_changes",
    "payments",
    "support_tickets"
]

print("[STAGE] Processing landing entities")

for entity in landing_entities:

    try:
        ingest = LandingToBronzeIngest(spark, entity)
        ingest.run()

    except Exception as e:
        print(f"[ERROR] Failed entity {entity}: {str(e)}")

print("[SUCCESS] Landing → Bronze completed")

# 3. Postgres → Bronze

In [ ]:
postgres_tables = [
    "public.license_keys",
    "public.license_allocations"
]

print("[STAGE] Processing Postgres tables")

for table in postgres_tables:

    table_short = table.split(".")[-1]
    bronze_path = f"/Volumes/datalake_catalog/datalake_schema/bronze/{table_short}"

    try:
        ingest = PostgresBronzeIngest(
            spark,
            table,
            bronze_path,
            dbutils=dbutils,
            batch_id=batch_id
        )
        ingest.run()

    except Exception as e:
        print(f"[ERROR] Failed table {table}: {str(e)}")

print("[SUCCESS] Postgres → Bronze completed")

# 4. Pipeline Completed

In [ ]:
print("[DONE] DAILY Bronze pipeline finished")